<a href="https://colab.research.google.com/github/giuliobarde/web_data_mining_project/blob/main/Mini_Project_2_RetrieveFrom_Pinecone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
pip install pinecone

In [17]:
# S3 to Pinecone Vector Database Pipeline
# Project for retrieving news articles from S3 and storing them in Pinecone vector database

import json
import boto3
import pandas as pd
from typing import List, Dict, Any, Optional
import pinecone
import time
from botocore.config import Config
from botocore import UNSIGNED

class NewsProcessor:
    """
    A class for processing news articles from S3 and storing them in Pinecone vector database.
    """

    def __init__(self, api_key: str, index_name: str, team_name: str, category: str):
        """
        Initialize the NewsProcessor with Pinecone credentials and team information.

        Args:
            api_key: Pinecone API key
            index_name: Pinecone index name
            team_name: Team name (e.g., 'Team 1')
            category: Assigned category (e.g., 'Health')
        """
        self.api_key = api_key
        self.index_name = index_name
        self.team_name = team_name
        self.category = category
        self.pinecone_client = None
        self.index = None

    def connect_to_pinecone(self) -> None:
        """Connect to Pinecone and initialize the index."""
        try:
            # Initialize Pinecone with API key
            # Try the newer initialization method first
            try:
                self.pinecone_client = pinecone.Pinecone(api_key=self.api_key)
                self.index = self.pinecone_client.Index(self.index_name)
            except AttributeError:
                # Fall back to older initialization method
                pinecone.init(api_key=self.api_key)
                self.index = pinecone.Index(self.index_name)

            print(f"Connected to Pinecone index: {self.index_name}")
        except Exception as e:
            print(f"Error connecting to Pinecone: {str(e)}")
            print("Please check your API key and index name.")
            raise


    def fetch_from_s3(self, bucket_name: str, team_folder: str, category: str = None, aws_access_key: str = None, aws_secret_key: str = None) -> List[Dict]:
        """
        Fetch news articles from S3 bucket, based on the structure observed in the notebook.

        Args:
            bucket_name: S3 bucket name
            team_folder: Team folder name (e.g., "TEAM_1/")
            category: Category to filter sources (optional)
            aws_access_key: AWS access key (optional if using IAM roles)
            aws_secret_key: AWS secret key (optional if using IAM roles)

        Returns:
            List of articles combined from all relevant source files
        """
        # Initialize S3 client - either with credentials or unsigned (anonymous)
        if aws_access_key and aws_secret_key:
            s3_client = boto3.client(
                's3',
                aws_access_key_id=aws_access_key,
                aws_secret_access_key=aws_secret_key
            )
        else:
            # Use unsigned access for public buckets or when using IAM roles
            s3_client = boto3.client('s3', config=Config(signature_version=UNSIGNED))

        try:
            # List all objects in the sources directory
            sources_prefix = f"{team_folder}sources/"
            print(f"Listing objects in s3://{bucket_name}/{sources_prefix}")
            response = s3_client.list_objects_v2(Bucket=bucket_name, Prefix=sources_prefix)

            if 'Contents' not in response:
                print(f"No source files found in s3://{bucket_name}/{sources_prefix}")
                return []

            # Filter sources based on category if provided
            source_files = []
            for obj in response['Contents']:
                file_key = obj['Key']
                # Skip if not a JSON file
                if not file_key.endswith('.json'):
                    continue

                # If category filter is applied, check source name
                if category:
                    source_name = file_key.split('/')[-1].replace('.json', '')
                    # Simple matching - can be improved based on actual categorization logic
                    if category.lower() in source_name.lower():
                        source_files.append(file_key)
                else:
                    source_files.append(file_key)

            print(f"Found {len(source_files)} source files")

            # Collect articles from all matching source files
            all_articles = []
            for file_key in source_files:
                try:
                    print(f"Fetching {file_key}")
                    response = s3_client.get_object(Bucket=bucket_name, Key=file_key)
                    file_content = response['Body'].read().decode('utf-8')
                    source_data = json.loads(file_content)

                    # The S3 files store articles with numeric keys
                    for article_id, article in source_data.items():
                        # Add source filename as metadata
                        article['source_file'] = file_key.split('/')[-1]
                        all_articles.append(article)

                    print(f"Added {len(source_data)} articles from {file_key}")
                except Exception as e:
                    print(f"Error processing {file_key}: {str(e)}")

            print(f"Total articles collected: {len(all_articles)}")
            return all_articles

        except Exception as e:
            print(f"Error fetching from S3: {str(e)}")
            # Provide additional guidance for common S3 issues
            if "AccessDenied" in str(e):
                print("Access denied. Check your IAM permissions or AWS credentials.")
            elif "NoSuchBucket" in str(e):
                print(f"Bucket '{bucket_name}' not found. Check the bucket name.")
            raise

    def process_articles(self, articles: List[Dict]) -> List[Dict]:
        """
        Process articles to prepare them for vector storage.

        Args:
            articles: List of article dictionaries

        Returns:
            List of processed articles with necessary metadata
        """
        processed_articles = []

        for i, article in enumerate(articles):
            # Extract required fields
            title = article.get('title', '')
            description = article.get('description', '')
            content = article.get('content', '')
            url = article.get('url', '')
            published_at = article.get('publishedAt', '')
            source_name = article.get('source', {}).get('name', '')
            source_file = article.get('source_file', 'unknown_source')
            author = article.get('author', '')

            # Combine text fields for embedding
            text_content = f"Title: {title}\nDescription: {description}\nContent: {content}"

            # Create document with metadata
            processed_article = {
                'id': f"{self.team_name}_{self.category}_{source_name}_{i}",
                'text': text_content,
                'metadata': {
                    'team': self.team_name,
                    'category': self.category,
                    'title': title,
                    'url': url,
                    'published_at': published_at,
                    'source': source_name,
                    'source_file': source_file,
                    'author': author
                }
            }

            processed_articles.append(processed_article)

        return processed_articles

    def upload_to_pinecone(self, processed_articles: List[Dict]) -> None:
        """
        Upload processed articles to Pinecone.

        Args:
            processed_articles: List of processed article dictionaries
        """
        if not self.index:
            raise ValueError("Not connected to Pinecone. Call connect_to_pinecone() first.")

        # Prepare records for upsert
        upsert_batch = []
        for article in processed_articles:
            # Note: No need to generate embeddings as the Pinecone index
            # automatically embeds documents using 'llama-text-embed-v2'
            upsert_batch.append({
                'id': article['id'],
                'values': None,  # Pinecone will auto-generate embeddings
                'metadata': {
                    'text': article['text'],
                    **article['metadata']
                }
            })

        # Upsert in batches of 100
        batch_size = 100
        for i in range(0, len(upsert_batch), batch_size):
            batch = upsert_batch[i:i+batch_size]

            try:
                # Try the newer API first
                try:
                    self.index.upsert(vectors=batch)
                except TypeError:
                    # Fall back to older API if needed
                    self.index.upsert(vectors=batch, namespace="")

                print(f"Uploaded batch {i//batch_size + 1}/{len(upsert_batch)//batch_size + 1}")
            except Exception as e:
                print(f"Error uploading batch {i//batch_size + 1}: {str(e)}")
                print("Continuing with next batch...")

        print(f"Upload complete. Total articles: {len(processed_articles)}")

    def query_pinecone(self, query_text: str, top_k: int = 5, filter_params: Dict = None) -> List[Dict]:
        """
        Query Pinecone for similar articles.

        Args:
            query_text: The text to query
            top_k: Number of results to return
            filter_params: Dictionary of filter parameters

        Returns:
            List of similar articles
        """
        if not self.index:
            raise ValueError("Not connected to Pinecone. Call connect_to_pinecone() first.")

        # Query the index
        # Note: No need to generate embeddings as Pinecone will auto-embed the query
        query_params = {
            'top_k': top_k,
            'include_metadata': True,
        }

        if filter_params:
            query_params['filter'] = filter_params

        try:
            # Try newer API format first
            try:
                results = self.index.query(
                    vector=None,  # Pinecone will auto-generate embedding
                    text=query_text,
                    **query_params
                )
            except TypeError:
                # Fall back to older API format
                if 'filter' in query_params:
                    filter_dict = query_params.pop('filter')
                    results = self.index.query(
                        text=query_text,
                        filter=filter_dict,
                        **query_params
                    )
                else:
                    results = self.index.query(
                        text=query_text,
                        **query_params
                    )

            return results.matches
        except Exception as e:
            print(f"Error querying Pinecone: {str(e)}")
            return []

    def analyze_category_insights(self) -> Dict:
        """
        Analyze category-specific insights from stored articles.

        Returns:
            Dictionary containing insights
        """
        # Query all documents from this team/category
        filter_params = {
            'team': self.team_name,
            'category': self.category
        }

        # Example queries specific to different categories
        category_queries = {
            'Health': ["medical research", "disease prevention", "healthcare policy", "wellness trends"],
            'Finance': ["stock market", "investment trends", "economic forecast", "financial news"],
            'Politics': ["election", "government policy", "political campaign", "international relations"],
            'Policies': ["regulation", "legislation", "policy reform", "government initiatives"],
            'World News': ["international events", "global crisis", "diplomatic relations", "world economy"],
            'Investment': ["stock performance", "investment strategy", "market analysis", "portfolio management"],
            'Leisure': ["entertainment", "sports events", "travel destinations", "recreational activities"]
        }

        # Get queries for this category or use default
        queries = category_queries.get(self.category, ["trending topics", "recent developments", "major events"])

        insights = {}
        for query in queries:
            results = self.query_pinecone(query, top_k=3, filter_params=filter_params)
            insights[query] = results

        return insights

# Example usage
if __name__ == "__main__":
    # Initialize processor with team information
    processor = NewsProcessor(
        api_key="pcsk_6akU8Z_2BXXXDSBKbvFCn4sciNM2FeJC6PwAt6wFwQeQjoJKDSjysRbtyBAdUfRv6z87e6",
        index_name="cus635",
        team_name="Team 1",  # Replace with your team number
        category="Finance"   # Replace with your assigned category
    )

    # Connect to Pinecone
    processor.connect_to_pinecone()

    # S3 bucket configuration - matches the structure shown in your notebook
    bucket_name = "cus635-spring2025"  # This is the actual bucket shown in your notebook
    team_folder = "TEAM_1/"  # Replace with your team folder

    try:
        # Fetch articles from S3
        print(f"Fetching articles from S3 bucket: {bucket_name}, team folder: {team_folder}")

        # Get all sources for your team
        articles = processor.fetch_from_s3(
            bucket_name=bucket_name,
            team_folder=team_folder
        )

        if not articles:
            print("No articles found. Please check your S3 bucket and team folder configuration.")
        else:
            # Process and upload to Pinecone
            processed_articles = processor.process_articles(articles)
            print(f"Processed {len(processed_articles)} articles, uploading to Pinecone...")
            processor.upload_to_pinecone(processed_articles)
            print("Upload complete!")

            # Query example
            print("\nPerforming sample query:")
            category_queries = {
                "Finance": ["stock market", "investment trends", "financial news"],
                "Health": ["medical research", "healthcare policy", "wellness trends"],
                "Politics": ["election", "government policy", "political debate"],
                "Policies": ["regulation", "legislation", "policy reform"],
                "World News": ["international relations", "global economy", "world events"],
                "Investment": ["stock market", "investment strategy", "portfolio management"],
                "Leisure": ["entertainment", "travel", "sports events"]
            }

            # Select queries related to your category
            relevant_queries = category_queries.get(processor.category, ["trending topics"])

            for query in relevant_queries:
                print(f"\nQuerying for: '{query}'")
                query_results = processor.query_pinecone(query, top_k=3)
                for i, result in enumerate(query_results, 1):
                    print(f"{i}. {result.metadata.get('title')} (Score: {result.score:.4f})")
                    print(f"   Source: {result.metadata.get('source')}")

            # Get category insights
            print("\nAnalyzing category insights:")
            insights = processor.analyze_category_insights()
            for query, results in insights.items():
                print(f"\nInsights for '{query}':")
                for i, result in enumerate(results, 1):
                    print(f"{i}. {result.metadata.get('title')} (Score: {result.score:.4f})")

    except Exception as e:
        print(f"Error: {str(e)}")
        print("If you're facing S3 access issues, ensure your AWS credentials are configured correctly.")

Connected to Pinecone index: cus635
Fetching articles from S3 bucket: cus635-spring2025, team folder: TEAM_1/
Listing objects in s3://cus635-spring2025/TEAM_1/sources/
Found 58 source files
Fetching TEAM_1/sources/ABC_News.json
Added 9 articles from TEAM_1/sources/ABC_News.json
Fetching TEAM_1/sources/ABC_News_AU_.json
Added 1 articles from TEAM_1/sources/ABC_News_AU_.json
Fetching TEAM_1/sources/ANSA_it.json
Added 1 articles from TEAM_1/sources/ANSA_it.json
Fetching TEAM_1/sources/AppleInsider.json
Added 1 articles from TEAM_1/sources/AppleInsider.json
Fetching TEAM_1/sources/Associated_Press.json
Added 2 articles from TEAM_1/sources/Associated_Press.json
Fetching TEAM_1/sources/BBC_News.json
Added 4 articles from TEAM_1/sources/BBC_News.json
Fetching TEAM_1/sources/BBC_Sport.json
Added 4 articles from TEAM_1/sources/BBC_Sport.json
Fetching TEAM_1/sources/Bild.json
Added 2 articles from TEAM_1/sources/Bild.json
Fetching TEAM_1/sources/Bleacher_Report.json
Added 2 articles from TEAM_1/